In [7]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import subprocess, xml.etree.ElementTree as ET, zipfile, re, os, shutil
import time, json
from textwrap import dedent
from collections import defaultdict

# ------------------------------- CONFIG ---------------------------------

@dataclass
class SLC_CFG:
    study_area: str      = "Monsanto"
    year_token: str      = "20200000"              # YYYY0000
    tile_code: str       = "A"
    country: str         = "PT"                    # PT -> 3763, ES -> 25830
    gpt_exe: str         = r"C:\Program Files\esa-snap\bin\gpt.exe"
    gpt_mem: str         = "8G"
    publish: bool        = True          # export single-band rasters
    nodata: float        = -9999.0       # Joao’s NDV

    inputs_dir: Path     = Path(r"M:\Project BLS\S1\SLC\Inputs")
    xml_dir: Path        = Path(r"M:\Project BLS\S1\XML\SLC\Coherence")
    work_root: Path      = Path(r"M:\Project BLS\S1\Work\SLC")
    out_root: Path       = Path(r"M:\Project BLS\S1\Exports\SLC")

    # cfg.subswath controls how subswaths are processed:
    # - "AUTO": automatically detect relevant subswath
    # - "IW1" / "IW2" / "IW3": force a single subswath
    # - "ALL": process all subswaths
    subswath: str = "IW3"   # "AUTO" / "IW1" / "IW2" / "IW3" / "ALL"
    
    slc_pols: list[str]  = ("VV","VH")                 # ("VV",) or ("VV","VH")

    do_unwrap: bool = True   # True => run SNAPHU + import + TC + GLCM
                             # False => stop after SnaphuExport (stage05)

    # NEW: multilook + Goldstein + SNAPHU
    ml_range: int        = 4       # nRgLooks
    ml_az: int           = 2       # nAzLooks (only used if you parameterise XML)
    gold_alpha: float    = 1.0     # keep for later if you add ${Alpha}
    gold_win: int        = 3       # same (if you add ${windowSizeString})
    snaphu_exe: str      = r"C:\Program Files\esa-snap\snaphu-v1.4.2_win64\bin\snaphu.exe"

    @property
    def epsg(self) -> str:
        return "EPSG:3763" if self.country.upper()=="PT" else "EPSG:25830"

cfg = SLC_CFG()

TC_WKT = {
    "EPSG:3763": """PROJCS["ETRS89 / Portugal TM06",
      GEOGCS["ETRS89",
        DATUM["European Terrestrial Reference System 1989",
          SPHEROID["GRS 1980",6378137.0,298.257222101,AUTHORITY["EPSG","7019"]],
          TOWGS84[0,0,0,0,0,0,0],AUTHORITY["EPSG","6258"]],
        PRIMEM["Greenwich",0.0,AUTHORITY["EPSG","8901"]],
        UNIT["degree",0.017453292519943295],AUTHORITY["EPSG","4258"]],
      PROJECTION["Transverse_Mercator",AUTHORITY["EPSG","9807"]],
      PARAMETER["central_meridian",-8.133108333333334],
      PARAMETER["latitude_of_origin",39.66825833333334],
      PARAMETER["scale_factor",1.0],
      PARAMETER["false_easting",0.0],
      PARAMETER["false_northing",0.0],
      UNIT["m",1.0],AXIS["Easting",EAST],AXIS["Northing",NORTH],
      AUTHORITY["EPSG","3763"]]""",

    "EPSG:25830": """PROJCS["ETRS89 / UTM zone 30N",
      GEOGCS["ETRS89",
        DATUM["European Terrestrial Reference System 1989",
          SPHEROID["GRS 1980",6378137.0,298.257222101,AUTHORITY["EPSG","7019"]],
          TOWGS84[0,0,0,0,0,0,0],AUTHORITY["EPSG","6258"]],
        PRIMEM["Greenwich",0.0,AUTHORITY["EPSG","8901"]],
        UNIT["degree",0.017453292519943295],AUTHORITY["EPSG","4258"]],
      PROJECTION["Transverse_Mercator",AUTHORITY["EPSG","9807"]],
      PARAMETER["central_meridian",-3.0],
      PARAMETER["latitude_of_origin",0.0],
      PARAMETER["scale_factor",0.9996],
      PARAMETER["false_easting",500000.0],
      PARAMETER["false_northing",0.0],
      UNIT["m",1.0],AXIS["Easting",EAST],AXIS["Northing",NORTH],
      AUTHORITY["EPSG","25830"]]""",
}

BURSTS = {
    "IW1": (1, 9),  # adjust if different
    "IW2": (1, 9),
    "IW3": (1, 9),
}

# ------------------------------ HELPERS ---------------------------------

XML_TEMPLATES = {
    "1_S1-SLC_Backgeocoding_GPT.xml": """\
    <graph id="S1_SLC_Backgeocoding">
  <version>1.0</version>

  <node id="ReadMaster">
    <operator>Read</operator>
    <parameters>
      <file>${Input_1}</file>
    </parameters>
  </node>

  <node id="ReadSlave">
    <operator>Read</operator>
    <parameters>
      <file>${Input_2}</file>
    </parameters>
  </node>

  <node id="SplitMaster">
    <operator>TOPSAR-Split</operator>
    <sources><sourceProduct refid="ReadMaster"/></sources>
    <parameters>
      <subswath>${Sub_Swath}</subswath>
      <firstBurstIndex>${First_Burst}</firstBurstIndex>
      <lastBurstIndex>${Last_Burst}</lastBurstIndex>
    </parameters>
  </node>

  <node id="SplitSlave">
    <operator>TOPSAR-Split</operator>
    <sources><sourceProduct refid="ReadSlave"/></sources>
    <parameters>
      <subswath>${Sub_Swath}</subswath>
      <firstBurstIndex>${First_Burst}</firstBurstIndex>
      <lastBurstIndex>${Last_Burst}</lastBurstIndex>
    </parameters>
  </node>

  <node id="OrbitMaster">
    <operator>Apply-Orbit-File</operator>
    <sources><sourceProduct refid="SplitMaster"/></sources>
    <parameters>
      <orbitType>Sentinel Precise (Auto Download)</orbitType>
      <polyDegree>3</polyDegree>
      <continueOnFail>false</continueOnFail>
    </parameters>
  </node>

  <node id="OrbitSlave">
    <operator>Apply-Orbit-File</operator>
    <sources><sourceProduct refid="SplitSlave"/></sources>
    <parameters>
      <orbitType>Sentinel Precise (Auto Download)</orbitType>
      <polyDegree>3</polyDegree>
      <continueOnFail>false</continueOnFail>
    </parameters>
  </node>

  <node id="BackGeocoding">
    <operator>Back-Geocoding</operator>
    <sources>
      <sourceProduct refid="OrbitMaster"/>
      <sourceProduct.1 refid="OrbitSlave"/>
    </sources>
    <parameters>
      <demName>SRTM 1Sec HGT</demName>
      <demResamplingMethod>BICUBIC_INTERPOLATION</demResamplingMethod>
      <resamplingType>BICUBIC_INTERPOLATION</resamplingType>
      <maskOutAreaWithoutElevation>true</maskOutAreaWithoutElevation>
    </parameters>
  </node>

  <node id="Write">
    <operator>Write</operator>
    <sources><sourceProduct refid="BackGeocoding"/></sources>
    <parameters>
      <file>${Output}</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>
""",

    "2_S1-SLC_ESD.xml": """\
    <graph id="S1_SLC_ESD">
  <version>1.0</version>

  <node id="Read">
    <operator>Read</operator>
    <parameters>
      <file>${Input_1}</file>
    </parameters>
  </node>

  <node id="ESD">
    <operator>Enhanced-Spectral-Diversity</operator>
    <sources><sourceProduct refid="Read"/></sources>
    <parameters>
      <fineWinWidthStr>512</fineWinWidthStr>
      <fineWinHeightStr>512</fineWinHeightStr>
      <fineWinAccAzimuth>16</fineWinAccAzimuth>
      <fineWinAccRange>16</fineWinAccRange>
      <fineWinOversampling>128</fineWinOversampling>
      <xCorrThreshold>0.1</xCorrThreshold>
      <cohThreshold>0.3</cohThreshold>
      <numBlocksPerOverlap>10</numBlocksPerOverlap>
      <esdEstimator>Periodogram</esdEstimator>
      <weightFunc>Inv Quadratic</weightFunc>
      <temporalBaselineType>Number of images</temporalBaselineType>
      <maxTemporalBaseline>4</maxTemporalBaseline>
      <integrationMethod>L1 and L2</integrationMethod>
      <doNotWriteTargetBands>false</doNotWriteTargetBands>
    </parameters>
  </node>

  <node id="Write">
    <operator>Write</operator>
    <sources><sourceProduct refid="ESD"/></sources>
    <parameters>
      <file>${Output}</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>
""",

    "3_S1-SLC_Deburst-Interferogram.xml": """\
    <graph id="S1_SLC_IFG_Deburst">
  <version>1.0</version>

  <node id="Read">
    <operator>Read</operator>
    <parameters>
      <file>${Input_1}</file>
    </parameters>
  </node>

  <node id="Interferogram">
    <operator>Interferogram</operator>
    <sources><sourceProduct refid="Read"/></sources>
    <parameters>
      <subtractFlatEarthPhase>true</subtractFlatEarthPhase>
      <srpPolynomialDegree>5</srpPolynomialDegree>
      <srpNumberPoints>501</srpNumberPoints>
      <orbitDegree>3</orbitDegree>
      <includeCoherence>true</includeCoherence>
      <cohWinAz>3</cohWinAz>
      <cohWinRg>10</cohWinRg>
      <squarePixel>true</squarePixel>
      <subtractTopographicPhase>true</subtractTopographicPhase>
      <demName>SRTM 1Sec HGT</demName>
      <tileExtensionPercent>100</tileExtensionPercent>
    </parameters>
  </node>

  <node id="Deburst">
    <operator>TOPSAR-Deburst</operator>
    <sources><sourceProduct refid="Interferogram"/></sources>
    <parameters/>
  </node>

  <node id="Write">
    <operator>Write</operator>
    <sources><sourceProduct refid="Deburst"/></sources>
    <parameters>
      <file>${Output}</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>
""",

    "4_S1-SLC_Goldstein-Multilook.xml": """\
    <graph id="S1_SLC_Goldstein_Multilook">
  <version>1.0</version>

  <node id="Read">
    <operator>Read</operator>
    <parameters>
      <file>${Input_1}</file>
    </parameters>
  </node>

  <node id="Multilook">
    <operator>Multilook</operator>
    <sources><sourceProduct refid="Read"/></sources>
    <parameters>
      <nRgLooks>${Range_Looks}</nRgLooks>
      <nAzLooks>${Azimuth_Looks}</nAzLooks>
      <outputIntensity>false</outputIntensity>
      <grSquarePixel>true</grSquarePixel>
    </parameters>
  </node>

  <node id="Goldstein">
    <operator>GoldsteinPhaseFiltering</operator>
    <sources><sourceProduct refid="Multilook"/></sources>
    <parameters>
      <alpha>${Gold_Alpha}</alpha>
      <FFTSizeString>64</FFTSizeString>
      <windowSizeString>${Gold_Window}</windowSizeString>
      <useCoherenceMask>false</useCoherenceMask>
      <coherenceThreshold>0.1</coherenceThreshold>
    </parameters>
  </node>

  <node id="Write">
    <operator>Write</operator>
    <sources><sourceProduct refid="Goldstein"/></sources>
    <parameters>
      <file>${Output}</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>
""",

    "5_S1-SLC_SnaphuExport.xml":"""\
    <graph id="Graph">
  <version>1.0</version>

  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>${Input_1}</file>
      <copyMetadata>true</copyMetadata>
    </parameters>
  </node>

  <node id="SnaphuExport">
    <operator>SnaphuExport</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <targetFolder>${Output}</targetFolder>
      <statCostMode>DEFO</statCostMode>
      <initMethod>MST</initMethod>
      <numberOfTileRows>10</numberOfTileRows>
      <numberOfTileCols>10</numberOfTileCols>
      <numberOfProcessors>4</numberOfProcessors>
      <rowOverlap>200</rowOverlap>
      <colOverlap>200</colOverlap>
      <tileCostThreshold>500</tileCostThreshold>
    </parameters>
  </node>
</graph>""",

    "6_S1-SLC_SnaphuImport.xml":"""\
    <graph id="Graph">
  <version>1.0</version>

  <node id="1-Read-Phase">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>${Input_1}</file>
      <copyMetadata>true</copyMetadata>
    </parameters>
  </node>

  <node id="2-Read-Unwrapped-Phase">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>${Input_2}</file>
      <copyMetadata>true</copyMetadata>
    </parameters>
  </node>

  <node id="3-SnaphuImport">
    <operator>SnaphuImport</operator>
    <sources>
      <sourceProduct refid="1-Read-Phase"/>
      <sourceProduct.1 refid="2-Read-Unwrapped-Phase"/>
    </sources>
    <parameters>
      <doNotKeepWrapped>false</doNotKeepWrapped>
    </parameters>
  </node>

  <node id="4-Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="3-SnaphuImport"/>
    </sources>
    <parameters>
      <file>${Output}</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>""",

    "7_S1-SLC_TerrainCorrection.xml":"""\
    <graph id="Graph">
  <version>1.0</version>

  <node id="Read">
    <operator>Read</operator>
    <parameters>
      <file>${Input_1}</file>
      <copyMetadata>true</copyMetadata>
    </parameters>
  </node>

  <node id="Terrain-Correction">
    <operator>Terrain-Correction</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <demName>SRTM 1Sec HGT</demName>
      <pixelSpacingInMeter>25.0</pixelSpacingInMeter>
      <mapProjection>${MapProjectionWKT}</mapProjection>
      <nodataValueAtSea>true</nodataValueAtSea>
      <saveSelectedSourceBand>true</saveSelectedSourceBand>
      <outputComplex>false</outputComplex>
      <applyRadiometricNormalization>false</applyRadiometricNormalization>
      <auxFile>Latest Auxiliary File</auxFile>
    </parameters>
  </node>

  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="Terrain-Correction"/>
    </sources>
    <parameters>
      <file>${Output}</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>
""",

    "8_S1-SLC_GLCM.xml": """\
    <graph id="S1_SLC_GLCM">
  <version>1.0</version>

  <node id="Read">
    <operator>Read</operator>
    <parameters>
      <file>${Input_1}</file>
    </parameters>
  </node>

  <node id="GLCM">
    <operator>GLCM</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <sourceBands>${Source_Bands}</sourceBands>
      <windowSizeStr>5x5</windowSizeStr>
      <angleStr>ALL</angleStr>
      <quantizerStr>Probabilistic Quantizer</quantizerStr>
      <quantizationLevelsStr>64</quantizationLevelsStr>
      <displacement>1</displacement>
      <noDataValue>-9999</noDataValue>
      <outputContrast>true</outputContrast>
      <outputDissimilarity>true</outputDissimilarity>
      <outputHomogeneity>true</outputHomogeneity>
      <outputASM>true</outputASM>
      <outputEnergy>true</outputEnergy>
      <outputMAX>true</outputMAX>
      <outputEntropy>true</outputEntropy>
      <outputMean>true</outputMean>
      <outputVariance>true</outputVariance>
      <outputCorrelation>true</outputCorrelation>
    </parameters>
  </node>

  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="GLCM"/>
    </sources>
    <parameters>
      <file>${Output}</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>
""",
}

METRICS = {
    "CONTRAST": "CONTRAST",
    "DISSIMILARITY": "DISSIMILARITY",
    "HOMOGENEITY": "HOMOGENEITY",   # keep as is
    "ASM": "ASM",
    "ENERGY": "ENERGY",
    "MAX": "MAX",
    "ENTROPY": "ENTROPY",
    "MEAN": "MEAN",
    "VARIANCE": "VARIANCE",
    "CORRELATION": "CORRELATION",
}

def resolve_raster_for_gdal(src: Path) -> Path:
    """
    If src is a .dim (BEAM-DIMAP), point GDAL to the main .img/.tif inside the .data dir.
    If src is already a raster (.tif, .img, etc.), return it unchanged.
    """
    src = Path(src)
    suf = src.suffix.lower()

    # Case 1: normal raster already
    if suf in (".tif", ".tiff", ".img"):
        return src

    # Case 2: BEAM-DIMAP header
    if suf == ".dim":
        data_dir = src.with_suffix(".data")
        cands = list(data_dir.glob("*.img")) + list(data_dir.glob("*.tif"))
        if not cands:
            raise FileNotFoundError(f"No IMG/TIF inside {data_dir} for {src.name}")
        # SNAP usually stores everything in one big .img – take the largest
        best = max(cands, key=lambda p: p.stat().st_size)
        print(f"[INFO] Using {best.name} (inside {data_dir.name}) for GDAL on {src.name}")
        return best

    # Anything else: just try it, or raise if you prefer strict
    return src


def _detect_pol(text: str) -> str:
    t = text.upper()
    if "_VV" in t or t.endswith("VV") or " VV " in t: return "VV"
    if "_VH" in t or t.endswith("VH") or " VH " in t: return "VH"
    # fall back to first requested pol if present, else VV
    return (cfg.slc_pols[0] if cfg.slc_pols else "VV").upper()

def _detect_token(desc: str) -> str:
    """
    Turn a SNAP band description into João token:
      - 'coh_IW2_VV_15Jun2020_27Jun2020'   -> COHVV
      - 'contrast_coh_IW2_VV_...'          -> COHVVCONTRAST
      - 'homogeneity_coh_IW2_VH_...'       -> COHVHHOMOGENEITY
    """
    d = (desc or "").strip()
    if not d:
        return "COH" + _detect_pol("")

    up = d.upper()
    pol = _detect_pol(up)

    # base coherence band?
    if up.startswith("COH") or "COH_" in up or "COHERENCE" in up:
        # see if a metric keyword is prefixed (GLCM outputs often prefix the metric)
        for k in METRICS:
            if up.startswith(k + "_") or f"_{k}_" in up:
                return f"COH{pol}{METRICS[k]}"
        return f"COH{pol}"

    # GLCM may produce names like 'CONTRAST_coh_...' etc.
    for k in METRICS:
        if up.startswith(k + "_") or f"_{k}_" in up:
            return f"COH{pol}{METRICS[k]}"

    # fallback: just COH<pol>
    return f"COH{pol}"


def run_gpt(graph_xml: Path, mem: str):
    t0 = time.time()
    cmd = [cfg.gpt_exe, str(graph_xml), "-c", mem]
    print("RUN:", " ".join(cmd))
    out = subprocess.run(cmd, capture_output=True, text=True)
    dt = time.time() - t0
    if out.returncode != 0:
        print("\nSTDOUT tail:\n", "\n".join(out.stdout.splitlines()[-60:]))
        print("\nSTDERR tail:\n", "\n".join(out.stderr.splitlines()[-60:]))
        raise RuntimeError(f"gpt failed: {graph_xml}")
    print(f"[OK] {graph_xml.name} in {dt:.1f}s")
    return out

def timed_publish_slc_glcm(glcm_dim: Path):
    t0 = time.time()
    publish_slc_glcm(glcm_dim)
    print(f"[OK] publish {glcm_dim.name} in {time.time()-t0:.1f}s")


def write_xml_from_template(template_xml: Path, out_xml: Path, mapping: dict[str,str]):
    # Minimal, safe placeholder fill for ${Name}
    txt = template_xml.read_text(encoding="utf-8")
    for k, v in mapping.items():
        txt = txt.replace("${"+k+"}", v)
    out_xml.parent.mkdir(parents=True, exist_ok=True)
    out_xml.write_text(txt, encoding="utf-8")
    return out_xml


_PLACEHOLDER_RX = re.compile(r"\$\{[^}]+\}")

def assert_no_placeholders(xml_text: str, *, name: str) -> None:
    missing = sorted(set(_PLACEHOLDER_RX.findall(xml_text)))
    if missing:
        raise RuntimeError(f"Unresolved placeholders in {name}: {missing}")

def assert_well_formed_xml(xml_text: str, *, name: str) -> None:
    try:
        ET.fromstring(xml_text)
    except Exception as e:
        raise RuntimeError(f"Malformed XML in {name}:\n{e}")

def write_graph(template_xml: str, out_xml: Path, **kw) -> Path:
    txt = template_xml
    for k, v in kw.items():
        txt = txt.replace(f"${{{k}}}", str(v))

    # 1) must have no ${...} left
    assert_no_placeholders(txt, name=out_xml.name)

    # 2) must be well-formed XML
    assert_well_formed_xml(txt, name=out_xml.name)

    out_xml.parent.mkdir(parents=True, exist_ok=True)
    out_xml.write_text(txt, encoding="utf-8")
    return out_xml

def get_xml(name: str) -> str:
    """
    Return XML template text by name.
    """
    if name not in XML_TEMPLATES:
        raise KeyError(f"XML template not embedded: {name}")
    return XML_TEMPLATES[name]

def list_safe_products(inputs_dir: Path):
    items = []
    for p in sorted(inputs_dir.iterdir()):
        u = p.name.upper()
        if (p.is_dir() and u.endswith(".SAFE")) or (p.is_file() and u.endswith(".ZIP")):
            items.append(p)
    return items

def _open_manifest(slcz: Path):
    # Return iterator over annotation XML path and its text
    if slcz.suffix.lower()==".zip":
        with zipfile.ZipFile(slcz) as z:
            for n in z.namelist():
                if n.lower().startswith("annotation/") and n.lower().endswith(".xml"):
                    with z.open(n) as f:
                        yield n, f.read().decode("utf-8", errors="ignore")
    else:
        ann = slcz / "annotation"
        for x in ann.glob("*.xml"):
            yield str(x.relative_to(slcz).as_posix()), x.read_text(encoding="utf-8", errors="ignore")

def detect_pols(p: Path) -> set[str]:
    """
    Infer polarisations from the Sentinel-1 filename token:
      _1SSV_ -> {'VV'}
      _1SSH_ -> {'HH'}
      _1SDV_ -> {'VV','VH'}
      _1SDH_ -> {'HH','HV'}
    Fallback: empty set if unknown.
    """
    u = p.name.upper()
    if "_1SDV_" in u: return {"VV","VH"}
    if "_1SDH_" in u: return {"HH","HV"}
    if "_1SSV_" in u: return {"VV"}
    if "_1SSH_" in u: return {"HH"}
    # Robust fallback via regex (handles future variants like 2S..):
    m = re.search(r"_\dS(D|S)([HV]{1,2})_", u)
    if m:
        polcode = m.group(2)
        if polcode == "VV": return {"VV"}
        if polcode == "HH": return {"HH"}
        if polcode == "DV": return {"VV","VH"}
        if polcode == "DH": return {"HH","HV"}
    return set()

def detect_subswaths_and_bursts(slcz: Path) -> dict[str, int]:
    """
    Approximate burst availability per subswath by counting annotation files.
    Returns {'IW1': n, 'IW2': n, 'IW3': n}
    """
    counts = {"IW1": 0, "IW2": 0, "IW3": 0}
    for name, _ in _open_manifest(slcz):
        m = re.search(r"iw([123])", name.lower())
        if m:
            counts[f"IW{m.group(1)}"] += 1
    return counts


def pick_common_subswath(master: Path, slave: Path) -> tuple[str, int, int]:
    """
    AUTO chooser: pick a subswath present in both products.
    Priority order: IW2, IW1, IW3 (your original).
    Bursts currently fixed to 1-9 (conservative).
    """
    cm = detect_subswaths_and_bursts(master)
    cs = detect_subswaths_and_bursts(slave)

    candidates = [(sw, cm[sw] + cs[sw]) for sw in ("IW2", "IW1", "IW3") if cm[sw] > 0 and cs[sw] > 0]
    if not candidates:
        return "IW2", 1, 9

    sw = max(candidates, key=lambda x: x[1])[0]
    return sw, 1, 9


def resolve_swath_request(master: Path, slave: Path) -> tuple[str, int, int]:
    """
    Single source of truth for (swath, firstBurst, lastBurst) given cfg.subswath mode.
    """
    if cfg.subswath in ("IW1", "IW2", "IW3"):
        fb, lb = BURSTS.get(cfg.subswath, (1, 9))
        return cfg.subswath, fb, lb

    if cfg.subswath == "AUTO":
        return pick_common_subswath(master, slave)

    raise ValueError(f"resolve_swath_request called with non-single mode: {cfg.subswath}")

def product_name(token: str) -> str:
    return f"{cfg.study_area}_{cfg.year_token}_S1SLC_{token}_{cfg.tile_code}"

def list_band_names(dim_path: Path):
    """
    Return band names for a SNAP BEAM-DIMAP product.

    1. Try to read names from the .dim XML.
    2. If that fails, fall back to *.img filenames in the .data folder
       (each .img = single band, stem used as band name).
    """
    dim_path = Path(dim_path)

    # --- 1) Try the .dim XML header ---
    names: list[str] = []
    try:
        root = ET.parse(dim_path).getroot()

        def add(n):
            if not n:
                return
            n = n.strip()
            if n and n not in names:
                names.append(n)

        # RasterDataNode blocks
        for node in root.findall(".//RasterDataNode"):
            for key, val in node.attrib.items():
                if key.lower() in ("name", "id"):
                    add(val)
            add(node.findtext("name"))
            add(node.findtext("Name"))
            add(node.findtext("id"))
            add(node.findtext("ID"))

        # Legacy <Band> blocks
        for node in root.findall(".//Band"):
            for key, val in node.attrib.items():
                if key.lower() in ("name", "id"):
                    add(val)
            add(node.findtext("name"))
            add(node.findtext("Name"))
            add(node.findtext("id"))
            add(node.findtext("ID"))
    except Exception:
        pass

    if names:
        return names

    # --- 2) Fallback: use *.img filenames in .data ---
    data_dir = dim_path.with_suffix(".data")
    if not data_dir.exists():
        raise RuntimeError(f"No .data folder next to {dim_path}")

    imgs = sorted(data_dir.glob("*.img"))
    if not imgs:
        raise RuntimeError(f"No .img files found in {data_dir}")

    # use stem as band name: 'coh_IW2_VV_...' etc.
    stems = [p.stem for p in imgs]
    return stems


def pick_pol_bands(band_names, pol):
    """
    From a list of names, pick the best matches for this polarisation (VV/VH).
    Returns dict like {'phase': 'Phase_ifg_IW2_VV_15Jun2020_27Jun2020',
                       'intensity': 'Intensity_ifg_IW2_VV_15Jun2020_27Jun2020_db',
                       'coh': 'coh_IW2_VV_15Jun2020_27Jun2020'}  (if present)
    """
    pl = pol.lower()

    def find_one(prefix, require_pol=True, extra_pred=lambda s: True):
        # “prefix” is e.g. 'phase', 'intensity', 'coh'
        # Accept a few common variants (e.g. intensity/amplitude)
        patterns = {
            'phase':    r'^phase',
            'intensity':r'^(intensity|amplitude)',
            'coh':      r'^(coh|coherence)'
        }
        rx = re.compile(patterns[prefix], re.I)
        cands = [bn for bn in band_names if rx.search(bn)]
        if require_pol:
            cands = [bn for bn in cands if f"_{pl}_" in bn.lower() or bn.lower().endswith(f"_{pl}")]
        # prefer names with date tokens (if multiple)
        cands.sort(key=lambda s: (("_"+pol+"_") not in s)*10 + ("ifg" not in s.lower())*5 + len(s))
        for bn in cands:
            if extra_pred(bn):
                return bn
        return None

    out = {
        'phase':     find_one('phase', require_pol=True),
        'intensity': find_one('intensity', require_pol=True, extra_pred=lambda s: True),
        'coh':       find_one('coh', require_pol=True)
    }
    return {k:v for k,v in out.items() if v}

def pick_fresh_product(stage_dir: Path, want: Path, search_roots: list[Path]) -> Path:
    """Return want if it exists; else search stage_dir first, then other roots for the freshest .dim/.tif."""
    # exact target?
    if want.exists():
        return want
    # same basename but tif?
    tif_alt = want.with_suffix(".tif")
    if tif_alt.exists():
        print(f"[WARN] Expected {want.name} but found {tif_alt.name}; using it.")
        return tif_alt

    # newest .dim/.tif in stage_dir
    cands = list(stage_dir.glob("*.dim")) + list(stage_dir.glob("*.tif"))
    if cands:
        cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        print(f"[WARN] Using freshest in {stage_dir.name}: {cands[0].name}")
        return cands[0]

    # broaden search (sometimes SNAP writes into input folder or parent)
    search = []
    for root in search_roots:
        search += list(root.rglob("*.dim"))
        search += list(root.rglob("*.tif"))
    if not search:
        raise FileNotFoundError(f"No Coherence output found in {stage_dir}")

    # prefer files whose name contains “S1SLC_COH” or “coh”
    def score(p: Path):
        n = p.name.lower()
        s = 0
        s += 100 if "s1slc_coh" in n else 0
        s += 50 if "coh" in n else 0
        s += int(p.stat().st_mtime)
        return s

    search.sort(key=score, reverse=True)
    print(f"[WARN] Expected {want.name}, using best match found: {search[0]}")
    return search[0]

def publish_tc_coherence(tc_dim: Path, swath: str, tile_code: str | None = None):
    tile_code = tile_code or cfg.tile_code
    outdir = cfg.out_root / f"SingleBands_{swath}"
    outdir.mkdir(parents=True, exist_ok=True)
    band_names = list_band_names(tc_dim)

    coh_bands = []
    for name in band_names:
        lname = name.lower()
        if lname.startswith("coh_"):
            if "_vv_" in lname:
                coh_bands.append(("VV", name))
            elif "_vh_" in lname:
                coh_bands.append(("VH", name))

    if not coh_bands:
        print("[WARN] No coherence bands found in TC.")
        return

    for pol, band in coh_bands:
        out_name = f"{cfg.study_area}_{cfg.year_token}_S1SLC_COH{pol}_{tile_code}.tif"
        out_path = outdir / out_name
        if out_path.exists():
            print("[SKIP] publish:", out_name)
            continue

        xml_text = f"""
<graph id="export_coh">
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <parameters><file>{tc_dim}</file></parameters>
  </node>
  <node id="BandSelect">
    <operator>BandSelect</operator>
    <sources><sourceProduct refid="Read"/></sources>
    <parameters><sourceBands>{band}</sourceBands></parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources><sourceProduct refid="BandSelect"/></sources>
    <parameters>
      <file>{out_path}</file>
      <formatName>GeoTIFF</formatName>
    </parameters>
  </node>
</graph>
"""

        run_xml = outdir / f"EXPORT_COH_{pol}_{tile_code}.xml"
        run_xml.write_text(xml_text, encoding="utf-8")

        # run SNAP, not GDAL
        run_gpt(run_xml, "8G")

        print("→ SNAP-export:", out_name)

def run_snaphu(snx_dir: Path) -> None:
    """
    Run external snaphu for each exported interferometric phase, using
    the SNAP-generated snaphu.conf.

    Expected structure:
      snx_dir/
        .../Monsanto_..._SNAPHU_A/
          Monsanto_..._S1SLC_GOLD_A/
            Phase_ifg_...snaphu.img
            Phase_ifg_...snaphu.img.hdr
            UnwPhase_ifg_...snaphu.img.hdr
            snaphu.conf
    """
    # 1) find snaphu.conf
    conf = next(snx_dir.rglob("snaphu.conf"), None)
    if not conf:
        print("[WARN] snaphu.conf not found anywhere in", snx_dir)
        return

    conf_dir = conf.parent
    print(f"[ OK ] Found snaphu.conf at: {conf}")

    # 2) find wrapped phase images in that directory
    phase_imgs = list(conf_dir.glob("Phase_ifg*.snaphu.img"))
    if not phase_imgs:
        print("[WARN] No Phase_ifg*.snaphu.img found in", conf_dir)
        return

    for img in phase_imgs:
        # Get number of columns from gdalinfo (instead of hardcoding 5363)
        info = json.loads(subprocess.check_output(
            ["gdalinfo", "-json", str(img)], text=True
        ))
        ncols = info["size"][0]

        print(f"[SNAPHU] Running in {conf_dir} : {img.name} (ncols={ncols})")

        cmd = [
            cfg.snaphu_exe,
            str(img.name),     # input phase file
            str(ncols),        # number of columns
            "-f", "snaphu.conf"
        ]
        # run inside conf_dir so outputs land there
        subprocess.run(cmd, cwd=conf_dir, check=True)

    # After this, you should have UnwPhase_ifg_...snaphu.img next to the headers

# ------------------------------ STAGES ----------------------------------

def stage01_backgeo(master: Path, slave: Path, subswath: str, first_burst: int, last_burst: int, out_dir: Path) -> Path:
    run_xml = out_dir / f"{cfg.study_area}_{cfg.year_token}_S1SLC_BACKGEO_{cfg.tile_code}_RUN.xml"
    out_dim = out_dir / f"{cfg.study_area}_{cfg.year_token}_S1SLC_BACKGEO_{cfg.tile_code}.dim"
    if out_dim.exists():
        print("[SKIP] Back-Geo exists:", out_dim.name)
        return out_dim

    # These placeholders are harmless if not present in the XML
    bg_xml = get_xml("1_S1-SLC_Backgeocoding_GPT.xml")
    
    write_graph(
        bg_xml, run_xml,
        Input_1=str(master),
        Input_2=str(slave),
        Sub_Swath=subswath,
        First_Burst=str(first_burst),
        Last_Burst=str(last_burst),
        Output=str(out_dim),
    )
    run_gpt(run_xml, "8G")
    return out_dim


def stage02_esd(backgeo_dim: Path, out_dir: Path) -> Path:
    run_xml = out_dir / f"{cfg.study_area}_{cfg.year_token}_S1SLC_ESD_{cfg.tile_code}_RUN.xml"
    out_dim = out_dir / f"{cfg.study_area}_{cfg.year_token}_S1SLC_ESD_{cfg.tile_code}.dim"
    if out_dim.exists():
        print("[SKIP] ESD exists:", out_dim.name)
        return out_dim

    esd_xml = get_xml("2_S1-SLC_ESD.xml")
    
    write_graph(
        esd_xml, run_xml,
        Input_1=str(backgeo_dim),
        Output=str(out_dim),
    )
    run_gpt(run_xml, "8G")
    return out_dim


def stage03_deburst_ifg(esd_dim: Path, out_dir: Path) -> Path:
    run_xml = out_dir / f"{cfg.study_area}_{cfg.year_token}_S1SLC_DEBURST_{cfg.tile_code}_RUN.xml"
    out_dim = out_dir / f"{cfg.study_area}_{cfg.year_token}_S1SLC_DEBURST_{cfg.tile_code}.dim"
    if out_dim.exists():
        print("[SKIP] Deburst exists:", out_dim.name)
        return out_dim

    db_xml = get_xml("3_S1-SLC_Deburst-Interferogram.xml")
    
    write_graph(
        db_xml, run_xml,
        Input_1=str(esd_dim),
        Output=str(out_dim),
    )
    run_gpt(run_xml, "8G")
    return out_dim

def stage04_goldstein_multilook(in_dim: Path, out_dir: Path) -> Path:
    out_dim = out_dir / f"{cfg.study_area}_{cfg.year_token}_S1SLC_GOLD_{cfg.tile_code}.dim"
    run_xml = out_dir / f"{cfg.study_area}_{cfg.year_token}_S1SLC_GOLD_{cfg.tile_code}_RUN.xml"

    if out_dim.exists():
        print(f"[SKIP] Goldstein+Multilook exists: {out_dim.name}")
        return out_dim

    gold_xml = get_xml("4_S1-SLC_Goldstein-Multilook.xml")
    
    write_graph(
    gold_xml, run_xml,
    Input_1=str(in_dim),
    Output=str(out_dim),
    Range_Looks=str(cfg.ml_range),
    Azimuth_Looks=str(cfg.ml_az),
    Gold_Alpha=str(cfg.gold_alpha),
    Gold_Window=str(cfg.gold_win),)

    run_gpt(run_xml, cfg.gpt_mem)
    return out_dim


def stage05_snaphu_export(in_dim: Path, out_dir: Path) -> Path:
    """
    Run SNAP SnaphuExport. This ONLY prepares:
      - Phase_ifg_*.snaphu (+ .hdr)
      - coh_*.snaphu (+ .hdr)
      - UnwPhase_ifg_*.snaphu.hdr  (header only, no data yet)
      - snaphu.conf

    It does NOT run the external snaphu binary.
    """
    snx_dir = out_dir / f"{cfg.study_area}_{cfg.year_token}_S1SLC_SNAPHU_{cfg.tile_code}"
    run_xml = out_dir / f"{snx_dir.name}_RUN.xml"

    if snx_dir.exists():
        print("[SKIP] SnaphuExport exists:", snx_dir.name)
        return snx_dir

    snx_xml = get_xml("5_S1-SLC_SnaphuExport.xml")

    write_graph(
        snx_xml, run_xml,
        Input_1=str(in_dim),
        Output=str(snx_dir),
    )
    run_gpt(run_xml, cfg.gpt_mem)
    return snx_dir

def stage06_snaphu_import(gold_dim: Path, snx_dir: Path, out_dir: Path) -> Path:
    """
    Import unwrapped phase from SNAPHU into the Goldstein+multilook product.

    gold_dim : .dim from stage04 (Goldstein+Multilook)
    snx_dir  : root folder from SnaphuExport (contains the GOLD_A subfolder)
    out_dir  : where to write the SNAPHUIMP .dim
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    out_dim = out_dir / f"{cfg.study_area}_{cfg.year_token}_S1SLC_SNAPHUIMP_{cfg.tile_code}.dim"
    run_xml = out_dir / f"{cfg.study_area}_{cfg.year_token}_S1SLC_SNAPHUIMP_{cfg.tile_code}_RUN.xml"

    if out_dim.exists():
        print("[SKIP] SnaphuImport exists:", out_dim.name)
        return out_dim

    snx_dir = Path(snx_dir)

    # 1) Look for the actual unwrapped *image* first (preferred)
    img_candidates = list(snx_dir.rglob("UnwPhase_ifg*.snaphu.img"))
    unw_path = None
    if img_candidates:
        unw_path = img_candidates[0]
        print("[ OK ] Using unwrapped phase IMAGE:", unw_path)
    else:
        # 2) Fallback: look for header, in case SNAP needs .hdr
        hdr_candidates = list(snx_dir.rglob("UnwPhase_ifg*.snaphu.hdr"))
        if hdr_candidates:
            unw_path = hdr_candidates[0]
            print("[ OK ] Using unwrapped phase HEADER:", unw_path)
        else:
            raise FileNotFoundError(
                f"No UnwPhase_ifg*.snaphu[.img/.hdr] found under {snx_dir}"
            )

    imp_xml = get_xml("6_S1-SLC_SnaphuImport.xml")

    write_graph(
        imp_xml,
        run_xml,
        Input_1=str(gold_dim),   # Goldstein+Multilook .dim
        Input_2=str(unw_path),   # Unwrapped phase (image or header)
        Output=str(out_dim),
    )
    run_gpt(run_xml, cfg.gpt_mem)
    return out_dim


def stage07_tc(in_dim: Path, out_dir: Path) -> Path:
    run_xml = out_dir / f"{cfg.study_area}_{cfg.year_token}_S1SLC_TC_{cfg.tile_code}_RUN.xml"
    out_dim = out_dir / f"{cfg.study_area}_{cfg.year_token}_S1SLC_TC_{cfg.tile_code}.dim"
    if out_dim.exists():
        print("[SKIP] TC exists:", out_dim.name)
        return out_dim

    tc_xml = get_xml("7_S1-SLC_TerrainCorrection.xml")

    map_wkt = TC_WKT[cfg.epsg]  # selects 3763 or 25830
    write_graph(tc_xml, run_xml,
                Input_1=str(in_dim),
                Output=str(out_dim),
                MapProjectionWKT=map_wkt)

    run_gpt(run_xml, cfg.gpt_mem)
    return out_dim

def stage08_glcm(tc_dim: Path, out_dir: Path) -> Path:
    out_dim = out_dir / f"{cfg.study_area}_{cfg.year_token}_S1SLC_GLCM_{cfg.tile_code}.dim"
    run_xml = out_dir / f"{cfg.study_area}_{cfg.year_token}_S1SLC_GLCM_{cfg.tile_code}_RUN.xml"
    
    if out_dim.exists():
        print("[SKIP] GLCM exists:", out_dim.name)
        return out_dim

    bands = list_band_names(tc_dim)
    coh_bands = [b for b in bands if b.lower().startswith("coh_")]
    if not coh_bands:
        raise RuntimeError(f"No coherence bands found in {tc_dim}")

    src_bands = ",".join(coh_bands)
    print("[INFO] Using coherence bands for GLCM:", coh_bands)

    glcm_xml = get_xml("8_S1-SLC_GLCM.xml")
    write_graph(
        glcm_xml, run_xml,
        Input_1=str(tc_dim),
        Source_Bands=src_bands,
        Output=str(out_dim),
    )
    run_gpt(run_xml, cfg.gpt_mem)
    print("[INFO] GLCM bands:", list_band_names(out_dim))
    return out_dim


def publish_slc_glcm(glcm_dim: Path, swath: str, tile_code: str | None = None):
    """
    Export GLCM textures derived from coherence bands.

    swath is explicit (IW1/IW2/IW3) so naming doesn't depend on cfg mutation.
    """
    glcm_dim = Path(glcm_dim)
    tile_code = tile_code or cfg.tile_code

    # output folder is swath-specific (consistent with coherence export)
    outdir = cfg.out_root / f"SingleBands_{swath}"
    outdir.mkdir(parents=True, exist_ok=True)

    band_names = list_band_names(glcm_dim)
    print("[INFO] GLCM bands:", band_names)

    data_dir = glcm_dim.with_suffix(".data")
    imgs = list(data_dir.glob("*.img")) + list(data_dir.glob("*.tif"))
    if not imgs:
        raise FileNotFoundError(f"No .img/.tif found in {data_dir}")

    multi_band = (len(imgs) == 1)
    if multi_band:
        multi_file = imgs[0]
        print(f"[INFO] Multi-band raster detected: {multi_file.name}")
    else:
        file_map = {p.stem: p for p in imgs}
        print(f"[INFO] Per-band rasters detected in {data_dir.name}: {len(file_map)} files")

    metric_map = [
        ("Contrast",        "CONTRAST"),
        ("Dissimilarity",   "DISSIMILARITY"),
        ("Homogeneity",     "HOMOGENEITY"),
        ("ASM",             "ASM"),
        ("Energy",          "ENERGY"),
        ("MAX",             "MAX"),
        ("Entropy",         "ENTROPY"),
        ("GLCMMean",        "MEAN"),
        ("GLCMVariance",    "VARIANCE"),
        ("GLCMCorrelation", "CORRELATION"),
    ]

    def detect_pol(name: str) -> str | None:
        n = name.lower()
        if "_vv_" in n: return "VV"
        if "_vh_" in n: return "VH"
        return None

    def detect_metric(name: str) -> str | None:
        lower = name.lower()
        for key, code in metric_map:
            if key.lower() in lower:
                return code
        return None

    for name in band_names:
        lname = name.lower()

        if "coh_" not in lname:
            continue

        pol = detect_pol(name)
        if pol is None:
            continue

        metric_code = detect_metric(name)
        if metric_code is None:
            continue

        # NOTE: use the explicit swath, not cfg.subswath
        token = f"COH{pol}{metric_code}_{swath}"
        out_name = f"{cfg.study_area}_{cfg.year_token}_S1SLC_{token}_{tile_code}.tif"
        out_path = outdir / out_name

        if out_path.exists():
            print("[SKIP] publish:", out_name)
            continue

        if multi_band:
            data_file = multi_file
            try:
                band_index = band_names.index(name) + 1  # 1-based
            except ValueError:
                print(f"[WARN] {name} not found in band_names; skipping.")
                continue
        else:
            data_file = file_map.get(name)
            if data_file is None:
                candidates = [p for p in imgs if p.stem.endswith(name)]
                if not candidates:
                    print(f"[WARN] No file found for band {name}; skipping.")
                    continue
                data_file = candidates[0]
            band_index = 1

        cmd = [
            "gdal_translate",
            str(data_file),
            str(out_path),
            "-b", str(band_index),
            "-a_nodata", str(cfg.nodata),
            "-co", "COMPRESS=LZW",
            "-co", "TILED=YES",
            "-co", "BIGTIFF=YES",
        ]
        print("RUN:", " ".join(cmd))
        subprocess.run(cmd, check=True)
        print("→ GDAL-export:", out_name)


# ---------------------------- ORCHESTRATOR ------------------------------
def run_slc_pair(swath: str, fb: int, lb: int, *, do_unwrap: bool | None = None) -> dict[str, Path]:
    """
    Run the SNAP SLC pipeline for ONE swath.

    If do_unwrap is False: stop after stage05 (SnaphuExport).
    If do_unwrap is True: run SNAPHU + Import + TC + GLCM.
    """
    if do_unwrap is None:
        do_unwrap = cfg.do_unwrap

    slcs = list_safe_products(cfg.inputs_dir)
    assert len(slcs) >= 2, "Need 2 SLC inputs (master & slave)"
    master, slave = slcs[0], slcs[1]

    pols = sorted(detect_pols(master) & detect_pols(slave))
    wanted = [p for p in cfg.slc_pols if p in pols]
    assert wanted, f"No requested pol in common. Available: {pols}"
    print("Pols:", wanted)
    print(f"Subswath: {swath}  Bursts: {fb}-{lb}")
    print(f"[MODE] do_unwrap={do_unwrap}")

    BG = cfg.work_root / f"01_Backgeo_{swath}"
    ESD = cfg.work_root / f"02_ESD_{swath}"
    DB = cfg.work_root / f"03_DeburstIFG_{swath}"
    GS = cfg.work_root / f"04_GoldsteinMultilook_{swath}"
    SE = cfg.work_root / f"05_SnaphuExport_{swath}"
    SI = cfg.work_root / f"06_SnaphuImport_{swath}"
    TC = cfg.work_root / f"07_TC_{swath}"
    GL = cfg.work_root / f"08_GLCM_{swath}"

    # always create dirs up to SE; others only needed if do_unwrap
    for d in (BG, ESD, DB, GS, SE, cfg.out_root):
        d.mkdir(parents=True, exist_ok=True)
    if do_unwrap:
        for d in (SI, TC, GL):
            d.mkdir(parents=True, exist_ok=True)

    backgeo = stage01_backgeo(master, slave, swath, fb, lb, BG)
    esd     = stage02_esd(backgeo, ESD)
    ifg     = stage03_deburst_ifg(esd, DB)
    gold    = stage04_goldstein_multilook(ifg, GS)
    snx_dir = stage05_snaphu_export(gold, SE)

    # ---- STOP HERE if unwrap disabled ----
    if not do_unwrap:
        return {"SNAPHU_EXPORT": snx_dir, "GOLD": gold}

    # ---- unwrap path ----
    run_snaphu(snx_dir)
    sn_imp  = stage06_snaphu_import(gold, snx_dir, SI)
    tc      = stage07_tc(sn_imp, TC)
    glcm    = stage08_glcm(tc, GL)

    return {"SNAPHU_EXPORT": snx_dir, "TC": tc, "GLCM": glcm}

def run_one(*, do_unwrap: bool | None = None) -> dict[str, Path]:
    slcs = list_safe_products(cfg.inputs_dir)
    assert len(slcs) >= 2, "Need 2 SLC inputs (master & slave)"
    master, slave = slcs[0], slcs[1]

    swath, fb, lb = resolve_swath_request(master, slave)
    return run_slc_pair(swath, fb, lb, do_unwrap=do_unwrap)

def run_all_subswaths(swaths=("IW1", "IW2", "IW3")):
    for swath in swaths:
        fb, lb = BURSTS.get(swath, (1, 9))
        print(f"\n=== PROCESSING {swath} ({fb}-{lb}) ===")

        result = run_slc_pair(swath, fb, lb, do_unwrap=cfg.do_unwrap)

        if cfg.publish and ("TC" in result) and ("GLCM" in result):
            publish_tc_coherence(result["TC"], swath)
            publish_slc_glcm(result["GLCM"], swath)
        else:
            if cfg.publish and not cfg.do_unwrap:
                print(f"[INFO] publish skipped for {swath} (do_unwrap=False).")

def main():
    if cfg.subswath == "ALL":
        print("[MODE] Running all subswaths: IW1, IW2, IW3")
        run_all_subswaths()
        return

    if cfg.subswath in ("AUTO", "IW1", "IW2", "IW3"):
        print(f"[MODE] {cfg.subswath}  do_unwrap={cfg.do_unwrap}")
        result = run_one(do_unwrap=cfg.do_unwrap)

        # Only publish if those products exist
        if cfg.publish and ("TC" in result) and ("GLCM" in result):
            # infer swath from TC folder if AUTO
            inferred_swath = result["TC"].parent.name.split("_")[-1]
            swath = inferred_swath if cfg.subswath == "AUTO" else cfg.subswath

            publish_tc_coherence(result["TC"], swath)
            publish_slc_glcm(result["GLCM"], swath)
        else:
            if cfg.publish and not cfg.do_unwrap:
                print("[INFO] publish skipped (do_unwrap=False => no TC/GLCM).")
        return

    raise ValueError(f"Invalid cfg.subswath value: {cfg.subswath}")


In [ ]:
if __name__ == "__main__":
    main()

[MODE] IW3  do_unwrap=True
Pols: ['VV', 'VH']
Subswath: IW3  Bursts: 1-9
[MODE] do_unwrap=True
RUN: C:\Program Files\esa-snap\bin\gpt.exe M:\Project BLS\S1\Work\SLC\01_Backgeo_IW3\Monsanto_20200000_S1SLC_BACKGEO_A_RUN.xml -c 8G
[OK] Monsanto_20200000_S1SLC_BACKGEO_A_RUN.xml in 921.5s
RUN: C:\Program Files\esa-snap\bin\gpt.exe M:\Project BLS\S1\Work\SLC\02_ESD_IW3\Monsanto_20200000_S1SLC_ESD_A_RUN.xml -c 8G
